# Training data

In [1]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [2]:
""" 
IMPORTS
"""
import os
import numpy as np
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --Machine learning and statistics

import pickle
from one.api import ONE
import concurrent.futures

# Get my functions
from functions import create_grouped_gradient_palette
# Get my functions
functions_path = prefix + '/representation_learning_variability/Functions/'
os.chdir(functions_path)
from one_functions_generic import download_subjectTables

one = ONE(mode='remote')

In [3]:
# Get my functions
def idxs_from_files(design_matrices):
    
    idxs = []
    mouse_names = []
    for m, mat in enumerate(design_matrices):
        mouse_name = design_matrices[m][51:]
        eid = design_matrices[m][14:50]
        idx = str(eid + '_' + mouse_name)

        if len(idxs) == 0:
            idxs = idx
            mouse_names = mouse_name
        else:
            idxs = np.hstack((idxs, idx))
            mouse_names = np.hstack((mouse_names, mouse_name))
            
    return idxs, mouse_names


""" 
LOAD DATA AND PARAMETERS
"""
# LOAD DATA

data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/'
# data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices'

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

learning_data_path =   prefix + 'representation_learning_variability/paper-individuality/data/training_data/'
# learning_data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/training_data/'


In [4]:
all_files = os.listdir(learning_data_path)
mice_to_download = []

for m, mouse_name in enumerate(np.unique(mouse_names)):
    
    filename = 'training_data_trials_'+mouse_name
    if filename not in all_files:
        mice_to_download.append(mouse_name)
    
print(len(mice_to_download))

102


# Download and save data

### When there are multiple versions take the most recent

In [5]:
import re
from pathlib import Path
from datetime import datetime

def get_latest_ibl_paths(path_list):
    """
    Filters a list of PosixPaths to return only the most recent patched versions.
    """
    latest_files = {}

    for path in path_list:
        path_str = str(path)
        
        # 1. Extract date from #YYYY-MM-DD# using regex
        match = re.search(r'#(\d{4}-\d{2}-\d{2})#', path_str)
        current_date = datetime.strptime(match.group(1), '%Y-%m-%d') if match else datetime.min
        
        # 2. Identify the unique dataset key (filename)
        # We strip out the UUID and the patch folder to identify "matching" datasets
        # e.g., 'CSHL052/_ibl_subjectTrials.table'
        dataset_key = path.name.split('.')[0] + "_" + path.parent.parent.name if match else path.name.split('.')[0] + "_" + path.parent.name
        
        # 3. Keep the one with the most recent date
        if dataset_key not in latest_files:
            latest_files[dataset_key] = (current_date, path)
        else:
            if current_date > latest_files[dataset_key][0]:
                latest_files[dataset_key] = (current_date, path)

    return [val[1] for val in latest_files.values()]



In [6]:
for m, mouse_name in enumerate(mice_to_download):
    # Trials data
    print(mouse_name)
    # Load only the DEFAULT (latest) revision via Flatiron. This avoids (1) the AWS-mirror gap
    # (the newest revision often isn't on the AWS bucket download_subjectTables reads from) and
    # (2) the multi-revision concatenation that used to duplicate every trial.
    lab = one.alyx.rest('subjects', 'list', nickname=mouse_name)[0]['lab']
    trials = one.load_aggregate('subjects', f'{lab}/{mouse_name}', '_ibl_subjectTrials.table.pqt')
    training = one.load_aggregate('subjects', f'{lab}/{mouse_name}', '_ibl_subjectTraining.table.pqt')
    training = training.reset_index()
    trained_date = pd.to_datetime(training.loc[training['training_status'].isin(['trained 1a', 'trained 1b']), 'date']).dt.date
    
    sessions = trials[['task_protocol', 'session_start_time', 'session']].dropna().drop_duplicates()
    training_sessions = sessions.loc[sessions['task_protocol'].str.contains('training')]
    trials['session_date'] = pd.to_datetime(trials['session_start_time']).dt.date
    mouse_data = trials.loc[trials['session_date']<np.min(trained_date)]

    # Save data
    mouse_data.to_parquet(learning_data_path+'training_data_trials_'+mouse_name,compression='gzip') 
    print('Successful for mouse ' + mouse_name)
    print(len(mouse_data))


CSHL045
Successful for mouse CSHL045
30599
CSHL047
Successful for mouse CSHL047
18477
CSHL049
Successful for mouse CSHL049
8763
CSHL051
Successful for mouse CSHL051
10394
CSHL052
Successful for mouse CSHL052
31083
CSHL053
Successful for mouse CSHL053
8600
CSHL054
Successful for mouse CSHL054
12069
CSHL055
Successful for mouse CSHL055
26845
CSHL058
Successful for mouse CSHL058
8552
CSHL059
Successful for mouse CSHL059
8842
CSHL060
Successful for mouse CSHL060
9966
CSH_ZAD_019
Successful for mouse CSH_ZAD_019
30546
CSH_ZAD_025
Successful for mouse CSH_ZAD_025
8656
CSH_ZAD_026
Successful for mouse CSH_ZAD_026
10897
CSH_ZAD_029
Successful for mouse CSH_ZAD_029
7837
DY_008
Successful for mouse DY_008
4122
DY_009
Successful for mouse DY_009
2009
DY_010
Successful for mouse DY_010
1097
DY_011
Successful for mouse DY_011
1538
DY_013
Successful for mouse DY_013
4175
DY_014
Successful for mouse DY_014
5115
DY_016
Successful for mouse DY_016
3417
DY_018
Successful for mouse DY_018
8856
DY_020
Suc